# 3장 예제:   

- 컨볼루션 합의 직접 계산(Algorithm 3.1), 그 결과를 numpy.convolve와 비교 검증,   
- 1차 IIR 차분방정식(지수 평활)의 재귀적 구현과 scipy.signal.lfilter 비교,   
- 그리고 이동평균 필터를 이용한 잡음 신호 평활화 응용까지를 포함한다.   

> 필요 라이브러리: numpy, scipy, matplotlib  (pip install numpy scipy matplotlib)


In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

In [ ]:

def discrete_convolution(x: np.ndarray, h: np.ndarray, nx0: int = 0, nh0: int = 0):
    """
    Algorithm 3.1을 그대로 구현한 컨볼루션 합 계산 함수.
    x, h: 1차원 배열 (각각 인덱스 nx0, nh0에서 시작한다고 가정)
    반환값: (y, ny0)  y는 결과 배열, ny0는 (3.19)에 따른 y의 시작 인덱스
    """
    Nx = len(x)
    Nh = len(h)
    Ny = Nx + Nh - 1
    ny0 = nx0 + nh0
    y = np.zeros(Ny)
    for i in range(Ny):
        k_min = max(0, i - Nh + 1)
        k_max = min(i, Nx - 1)
        acc = 0.0
        for k in range(k_min, k_max + 1):
            acc += x[k] * h[i - k]
        y[i] = acc
    return y, ny0


def moving_average_filter(M: int) -> np.ndarray:
    """길이 M의 이동평균 필터(FIR 임펄스응답)를 반환한다. h[n] = 1/M, n = 0,...,M-1 (식 3.22)."""
    return np.ones(M) / M


def exponential_smoothing_iir(x: np.ndarray, alpha: float, y_init: float = 0.0) -> np.ndarray:
    """
    1차 IIR 차분방정식 y[n] = alpha*y[n-1] + (1-alpha)*x[n] (식 3.23)을
    재귀적으로 직접 계산한다. y_init은 초기조건 y[-1]에 해당한다.
    """
    y = np.zeros_like(x, dtype=float)
    y_prev = y_init
    for n in range(len(x)):
        y[n] = alpha * y_prev + (1 - alpha) * x[n]
        y_prev = y[n]
    return y


if __name__ == "__main__":
    # ------------------------------------------------------------------
    # (1) 3.3.2절 손계산 예시 검증: x=[1,2,3], h=[1,1,1]
    # ------------------------------------------------------------------
    x_demo = np.array([1.0, 2.0, 3.0])
    h_demo = np.array([1.0, 1.0, 1.0])
    y_algo, ny0_demo = discrete_convolution(x_demo, h_demo)
    y_ref = np.convolve(x_demo, h_demo, mode='full')
    print("=== (1) 컨볼루션 손계산 예시 검증 ===")
    print(f"Algorithm 3.1 결과 : {y_algo}  (시작 인덱스 n={ny0_demo})")
    print(f"np.convolve 결과   : {y_ref}")
    print(f"두 결과 일치 여부   : {np.allclose(y_algo, y_ref)}")

    # ------------------------------------------------------------------
    # (2) 1차 IIR 차분방정식: 직접 재귀 계산 vs scipy.signal.lfilter
    # ------------------------------------------------------------------
    np.random.seed(0)
    n = np.arange(0, 200)
    fs = 200.0
    f0 = 3.0
    clean_signal = np.sin(2 * np.pi * f0 * n / fs)
    noisy_signal = clean_signal + 0.4 * np.random.randn(len(n))

    alpha = 0.9
    y_iir_direct = exponential_smoothing_iir(noisy_signal, alpha)
    b_coef = [1 - alpha]
    a_coef = [1, -alpha]
    y_iir_scipy = signal.lfilter(b_coef, a_coef, noisy_signal)
    print("\n=== (2) IIR 차분방정식(지수 평활) 검증 ===")
    print(f"직접 재귀 계산과 scipy.signal.lfilter 결과 일치 여부: "
          f"{np.allclose(y_iir_direct, y_iir_scipy)}")

    # ------------------------------------------------------------------
    # (3) FIR 이동평균 필터를 이용한 잡음 평활화 (3.8절 응용 사례에서 해석)
    # ------------------------------------------------------------------
    M = 9
    h_ma = moving_average_filter(M)
    y_ma, ny0_ma = discrete_convolution(noisy_signal, h_ma)
    n_ma = np.arange(ny0_ma, ny0_ma + len(y_ma))

    # ------------------------------------------------------------------
    # (4) 시각화
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(3, 1, figsize=(9, 10), facecolor='white')
    for ax in axes:
        ax.set_facecolor('white')
        ax.grid(True, linewidth=0.5, alpha=0.5)

    axes[0].plot(n, noisy_signal, color='0.6', linewidth=0.9, label='Noisy input x[n]')
    axes[0].plot(n, clean_signal, color='black', linewidth=1.3, linestyle='--',
                 label='Clean reference')
    axes[0].set_title('Input signal (noisy sinusoid)')
    axes[0].set_xlabel('Index n')
    axes[0].set_ylabel('Amplitude')
    axes[0].annotate('Noisy input signal', xy=(40, noisy_signal[40]),
                      xytext=(60, 2.0), arrowprops=dict(arrowstyle='->', color='gray'))
    axes[0].legend(loc='upper right')

    axes[1].plot(n, noisy_signal, color='0.75', linewidth=0.9, label='Noisy input x[n]')
    axes[1].plot(n_ma, y_ma, color='tab:blue', linewidth=1.6,
                 label=f'FIR moving-average output (M={M})')
    axes[1].set_title('FIR moving-average filtering: y[n] = x[n] * h[n]')
    axes[1].set_xlabel('Index n')
    axes[1].set_ylabel('Amplitude')
    axes[1].annotate('FIR (convolution) output', xy=(100, y_ma[100]),
                      xytext=(120, -1.8), arrowprops=dict(arrowstyle='->', color='tab:blue'))
    axes[1].legend(loc='upper right')

    axes[2].plot(n, noisy_signal, color='0.75', linewidth=0.9, label='Noisy input x[n]')
    axes[2].plot(n, y_iir_direct, color='tab:red', linewidth=1.6,
                 label=f'IIR exponential-smoothing output (alpha={alpha})')
    axes[2].set_title('IIR difference-equation filtering: y[n] = a*y[n-1] + (1-a)*x[n]')
    axes[2].set_xlabel('Index n')
    axes[2].set_ylabel('Amplitude')
    axes[2].annotate('IIR (difference-equation) output', xy=(100, y_iir_direct[100]),
                      xytext=(120, -1.8), arrowprops=dict(arrowstyle='->', color='tab:red'))
    axes[2].legend(loc='upper right')

    fig.tight_layout()
    plt.show()